# Steps 4–5: Results Routing & Clinical Follow-Up Pathways

This notebook covers what happens **after** a screening result is returned:

- **Step 4 — Result routing**
  - Cervical: NORMAL → surveillance; abnormal → colposcopy referral (with LTFU check);
    HPV+/normal cyto → 1-year repeat or colposcopy.
  - Other cancers: NEGATIVE → surveillance; POSITIVE → stub referral pathway.

- **Step 5 — Follow-up execution**
  - Colposcopy → CIN grade draw → LEEP / cone / surveillance.
  - Loss-to-follow-up modelled explicitly at two nodes.

All logic lives in `followup.py`. This notebook demonstrates and tests it.

In [1]:
import sys, random
sys.path.insert(0, '../src')

import config as cfg
from patient import Patient
from population import sample_patient
from followup import (
    route_cervical_result,
    run_colposcopy,
    run_treatment,
    run_cervical_followup,
    run_stub_followup,
)
from metrics import initialize_metrics

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

ImportError: cannot import name 'run_stub_followup' from 'followup' (/Users/alexandrapaiz/Desktop/NYP/notebooks/../src/followup.py)

## Result Routing — Cervical
Show which pathway each result category triggers.

In [2]:
results_to_test = [
    'NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POS_NORMAL_CYTO'
]

# Run 200 trials per result to show routing probabilities (LTFU is stochastic)
from collections import Counter

print(f'{"Result":<25}  Routing distribution (200 trials)')
print('-' * 70)
for result in results_to_test:
    routes = []
    for _ in range(200):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, result
        routes.append(route_cervical_result(p, day := 0))
    cnt = Counter(routes)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {result:<23}  {summary}')

Result                     Routing distribution (200 trials)
----------------------------------------------------------------------
  NORMAL                   routine_surveillance=200
  ASCUS                    colposcopy=151  exit=49
  LSIL                     colposcopy=162  exit=38
  ASC-H                    colposcopy=163  exit=37
  HSIL                     colposcopy=162  exit=38
  HPV_POS_NORMAL_CYTO      routine_surveillance=200


## Colposcopy Result Distribution
Verify CIN grade draws match config probabilities for each triggering result.

In [3]:
triggering_results = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POS_NORMAL_CYTO']
N = 1_000

for trigger in triggering_results:
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix, p.cervical_result = 42, True, trigger

    cin_counts = Counter(
        run_colposcopy(p.__class__(**{**p.__dict__, 'cervical_result': trigger,
                                      'event_log': []}), 0)
        for _ in range(N)
    )
    expected = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{trigger}', {})
    print(f'\nColposcopy from {trigger} (n={N:,}):')
    for grade in ['NORMAL', 'CIN1', 'CIN2', 'CIN3']:
        obs  = cin_counts.get(grade, 0) / N * 100
        exp  = expected.get(grade, 0) * 100
        print(f'  {grade:<8} observed={obs:5.1f}%  expected={exp:5.1f}%')


Colposcopy from ASCUS (n=1,000):
  NORMAL   observed= 60.4%  expected= 60.0%
  CIN1     observed= 25.5%  expected= 25.0%
  CIN2     observed=  9.6%  expected= 10.0%
  CIN3     observed=  4.5%  expected=  5.0%

Colposcopy from LSIL (n=1,000):
  NORMAL   observed= 41.2%  expected= 40.0%
  CIN1     observed= 33.1%  expected= 35.0%
  CIN2     observed= 15.8%  expected= 15.0%
  CIN3     observed=  9.9%  expected= 10.0%

Colposcopy from ASC-H (n=1,000):
  NORMAL   observed= 24.7%  expected= 25.0%
  CIN1     observed= 21.6%  expected= 20.0%
  CIN2     observed= 28.6%  expected= 30.0%
  CIN3     observed= 25.1%  expected= 25.0%

Colposcopy from HSIL (n=1,000):
  NORMAL   observed=  9.0%  expected= 10.0%
  CIN1     observed= 10.2%  expected= 10.0%
  CIN2     observed= 30.8%  expected= 30.0%
  CIN3     observed= 50.0%  expected= 50.0%

Colposcopy from HPV_POS_NORMAL_CYTO (n=1,000):
  NORMAL   observed= 48.6%  expected=  0.0%
  CIN1     observed= 25.1%  expected=  0.0%
  CIN2     observed= 14.6%

## Full Cervical Follow-Up — End-to-End Trace
Run 10 patients from screening result → colposcopy → treatment/surveillance.
Print each patient's event log.

In [4]:
from screening import run_screening_step, get_eligible_screenings

random.seed(99)
DAY = 0
metrics = initialize_metrics()

n_run = 0
traced = []
pid = 0

while n_run < 10:
    p = sample_patient(pid, DAY, 'gynecologist', 'outpatient')
    pid += 1
    if 'cervical' not in get_eligible_screenings(p):
        continue

    result = run_screening_step(p, 'cervical', DAY)
    if result is None:
        continue

    disposition = run_cervical_followup(p, DAY, metrics)
    p.log(DAY, f'DISPOSITION: {disposition}')
    traced.append(p)
    n_run += 1

# Print traces
for p in traced:
    p.print_history()

NameError: name 'initialize_metrics' is not defined

## Disposition Summary

In [5]:
from metrics import print_summary

# Populate metrics from traced patients
metrics['n_patients'] = len(traced)
metrics['n_eligible_any'] = len(traced)
for p in traced:
    metrics['n_screened']['cervical'] += 1
    if p.cervical_result:
        metrics['cervical_results'][p.cervical_result] += 1
    if p.exit_reason:
        from metrics import record_exit
        record_exit(metrics, p.exit_reason)

print_summary(metrics)

NameError: name 'traced' is not defined